In [1]:
import pandas as pd

voter = "Data/nsn_se15_2023.csv"

df = pd.read_csv(voter)
print(df.shape)   # (864425, 11)
print(df.dtypes)
df.head(5)

(864425, 11)
uid           object
birth_year     int64
sex           object
ethnicity     object
state         object
parlimen      object
dun           object
dm_vr         object
dm            object
pm            object
saluran        int64
dtype: object


,uid,birth_year,sex,ethnicity,state,parlimen,dun,dm_vr,dm,pm,saluran
0,VOKNZLPHGP,1963,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/05 Pekan Titi,126/01/00 Undi Awal,Khemah A IPD Jelebu,1
1,WHCVNPJUCD,1965,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/05 Pekan Titi,126/01/00 Undi Awal,Khemah A IPD Jelebu,1
2,YYBWEDQIUK,1966,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/03 Simpang Durian,126/01/00 Undi Awal,Khemah A IPD Jelebu,1
3,FXWUHPWPPG,1968,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/05 Pekan Titi,126/01/00 Undi Awal,Khemah A IPD Jelebu,1
4,VQKCMMVLKX,1976,Male,Malay,Negeri Sembilan,P.126 JELEBU,N01 CHENNAH,126/01/03 Simpang Durian,126/01/00 Undi Awal,Khemah A IPD Jelebu,1


In [2]:
results = "Data/ns_prn2023_results.csv"

df2 = pd.read_csv(results)
print(df2.shape)   # (864425, 11)
# print(df2.dtypes)
df2.head(5)

(1827, 18)


,PARLIMEN,DUN,NO. KOD DAERAH MENGUNDI,NAMA PUSAT MENGUNDI,SALURAN,KERTAS UNDI DALAM PETI UNDI (A),PN,PH,BN,MUDA,IND,IND.1,JUMLAH UNDI,UNDI YANG DITOLAK (C),KERTAS UNDI TIDAK DIMASUKKAN KE DALAM PETI UNDI (D),Unnamed: 15,Unnamed: 16,Unnamed: 17
0,P.126 JELEBU,N01 CHENNAH,UNDI POS,NaN,NaN,115,64,29,0,0,0,0,93,8,14,NaN,NaN,NaN
1,P.126 JELEBU,N01 CHENNAH,UNDI AWAL 126 / 01 / 00,KHEMAH A IPD JELEBU,1.0,29,20,9,0,0,0,0,29,0,0,NaN,NaN,NaN
2,P.126 JELEBU,N01 CHENNAH,KAMPONG SUNGAI BULOH 126 / 01 / 01,SEKOLAH KEBANGSAAN SUNGAI BULOH,1.0,325,165,160,0,0,0,0,325,0,0,NaN,NaN,NaN
3,P.126 JELEBU,N01 CHENNAH,KAMPONG SUNGAI BULOH 126 / 01 / 01,SEKOLAH KEBANGSAAN SUNGAI BULOH,2.0,350,229,118,0,0,0,0,347,3,0,NaN,NaN,NaN
4,P.126 JELEBU,N01 CHENNAH,DURIAN TIPUS 126 / 01 / 02,SEKOLAH KEBANGSAAN DURIAN TIPUS,1.0,267,73,192,0,0,0,0,265,2,0,NaN,NaN,NaN


## Aggregate the PRN2023 roll into ethnic composition per polling stream (saluran)

Drop postal/early voters, extract a clean station code, bucket ethnicity into
Malay/Chinese/Indian/Other, and compute each stream's ethnic composition + registered
voter count.

In [3]:
roll = df.copy()

# Drop postal and early voters
is_postal_or_early = roll['dm'].str.contains('Undi Pos|Undi Awal', na=False)
roll_ordinary = roll[~is_postal_or_early].copy()

print(f"dropped {is_postal_or_early.sum()} postal/early voters "
      f"({is_postal_or_early.sum() / len(roll) * 100:.2f}% of {len(roll)})")

# Extract a clean station code from dm, e.g. "126/01/01 Kampong Sungai Buloh" -> "126/01/01"
roll_ordinary['dm_code'] = (
    roll_ordinary['dm']
    .str.extract(r'^(\d+\s*/\s*\d+\s*/\s*\d+)')[0]
    .str.replace(r'\s+', '', regex=True)
)
assert roll_ordinary['dm_code'].isna().sum() == 0, "some ordinary rows failed dm_code extraction"

# Bucket ethnicity into the paper's four categories (Orang Asli / Bumi Sabah / Bumi / Sarawak fold into "Other")
ethnicity_map = {
    'Malay': 'malay',
    'Chinese': 'chinese',
    'Indian': 'indian',
    'Other': 'other',
    'Orang Asli': 'other',
    'Bumi Sabah': 'other',
    'Bumi Sarawak': 'other',
}
roll_ordinary['No.'] = roll_ordinary['ethnicity'].map(ethnicity_map)
assert roll_ordinary['No.'].isna().sum() == 0, "unmapped ethnicity value found"

# Group by (dun, dm_code, saluran) and count voters per ethnic group
stream_counts = (
    roll_ordinary
    .groupby(['dun', 'dm_code', 'saluran', 'No.'])
    .size()
    .unstack('No.', fill_value=0)
)

# Convert to percentages + keep registered count
stream_ethnic = stream_counts.copy()
stream_ethnic['n_registered'] = stream_counts.sum(axis=1)
for col in ['malay', 'chinese', 'indian', 'other']:
    stream_ethnic[f'pct_{col}'] = stream_counts[col] / stream_ethnic['n_registered']

stream_ethnic = stream_ethnic.reset_index()

# Check percentages sum to ~1.0
pct_cols = [f'pct_{c}' for c in ['malay', 'chinese', 'indian', 'other']]
pct_sum = stream_ethnic[pct_cols].sum(axis=1)
bad = stream_ethnic[(pct_sum - 1).abs() > 0.02]
assert len(bad) == 0, f"{len(bad)} streams have ethnic percentages not summing to ~1.0"

print(stream_ethnic.shape)
stream_ethnic.head(5)

dropped 26225 postal/early voters (3.03% of 864425)


(1405, 12)


No.,dun,dm_code,saluran,chinese,indian,malay,other,n_registered,pct_malay,pct_chinese,pct_indian,pct_other
0,N01 CHENNAH,126/01/01,1,1,0,447,2,450,0.993333,0.002222,0.000000,0.004444
1,N01 CHENNAH,126/01/01,2,0,0,535,4,539,0.992579,0.000000,0.000000,0.007421
2,N01 CHENNAH,126/01/02,1,248,17,149,36,450,0.331111,0.551111,0.037778,0.080000
3,N01 CHENNAH,126/01/02,2,55,8,124,156,343,0.361516,0.160350,0.023324,0.454810
4,N01 CHENNAH,126/01/03,1,123,22,302,3,450,0.671111,0.273333,0.048889,0.006667


## A4: Melt PRN2023 results into long format and merge with ethnic composition

Turn the wide `PN`/`PH`/`BN`/`MUDA`/`IND`/`IND.1` columns into one row per
party-per-stream, drop postal/early rows (same exclusion as A3), compute each row's
`vote_share`, then join against `stream_ethnic` on `(dun, dm_code, saluran)`.

In [4]:
results = df2.copy()
results['DUN'] = results['DUN'].str.strip()

# Drop postal and early rows (same exclusion as A3, see CLAUDE.md for why)
is_postal = results['NO. KOD DAERAH MENGUNDI'] == 'UNDI POS'
is_early = results['NO. KOD DAERAH MENGUNDI'].str.contains('UNDI AWAL', na=False)
results_ordinary = results[~(is_postal | is_early)].copy()

print(f"dropped {(is_postal | is_early).sum()} postal/early rows out of {len(results)}")

dropped 116 postal/early rows out of 1827


In [5]:
# Extract a clean station code from the messy "NAME 126 / 01 / 01" text
# (\s already matches the embedded newlines noted in CLAUDE.md)
code_parts = results_ordinary['NO. KOD DAERAH MENGUNDI'].str.extract(r'(\d+)\s*/\s*(\d+)\s*/\s*(\d+)')
results_ordinary['dm_code'] = code_parts[0] + '/' + code_parts[1] + '/' + code_parts[2]
assert results_ordinary['dm_code'].isna().sum() == 0, "some ordinary rows failed dm_code extraction"

results_ordinary['saluran'] = results_ordinary['SALURAN'].astype(int)

In [6]:
# Some stations report multiple "meja" (ballot box) rows for the same saluran when a
# stream is large enough to need more than one box - e.g. Kampung Serting Ilir's
# saluran 1 is split across two rows. Sum these up to one row per physical stream so
# the granularity matches stream_ethnic (one row per (dun, dm_code, saluran)).
num_cols = ['PN', 'PH', 'BN', 'MUDA', 'IND', 'IND.1', 'JUMLAH UNDI',
            'KERTAS UNDI DALAM PETI UNDI (A)', 'UNDI YANG DITOLAK (C)']
results_collapsed = (
    results_ordinary
    .groupby(['DUN', 'dm_code', 'saluran'], as_index=False)[num_cols]
    .sum()
)

print(f"collapsed {len(results_ordinary)} rows -> {len(results_collapsed)} streams "
      f"(expect 1405, matching stream_ethnic)")
assert len(results_collapsed) == len(stream_ethnic), "stream count mismatch after collapsing meja rows"

collapsed 1711 rows -> 1405 streams (expect 1405, matching stream_ethnic)


In [7]:
# Melt party columns into long form. A "0" already means "didn't contest here" (fixed
# columns are populated with 0, not left blank/NaN), so filter votes > 0 after melting.
# NOTE: IND and IND.1 are two DISTINCT independents where a seat has both (N10 Nilai,
# N13 Sikamat) - kept as separate "party" labels here, not summed together.
party_cols = ['PN', 'PH', 'BN', 'MUDA', 'IND', 'IND.1']
results_long = results_collapsed.melt(
    id_vars=['DUN', 'dm_code', 'saluran', 'JUMLAH UNDI'],
    value_vars=party_cols,
    var_name='party',
    value_name='votes',
)
results_long = results_long[results_long['votes'] > 0].copy()
results_long = results_long.rename(columns={'DUN': 'dun', 'JUMLAH UNDI': 'valid_votes'})
results_long['vote_share'] = results_long['votes'] / results_long['valid_votes']

print(results_long.shape)
results_long.head(10)

(3227, 7)


,dun,dm_code,saluran,valid_votes,party,votes,vote_share
0,N01 CHENNAH,126/01/01,1,325,PN,165,0.507692
1,N01 CHENNAH,126/01/01,2,347,PN,229,0.659942
2,N01 CHENNAH,126/01/02,1,265,PN,73,0.275472
3,N01 CHENNAH,126/01/02,2,215,PN,76,0.353488
4,N01 CHENNAH,126/01/03,1,282,PN,98,0.347518
5,N01 CHENNAH,126/01/03,2,344,PN,131,0.380814
6,N01 CHENNAH,126/01/03,3,329,PN,162,0.492401
7,N01 CHENNAH,126/01/03,4,290,PN,142,0.489655
8,N01 CHENNAH,126/01/04,1,309,PN,112,0.362460
9,N01 CHENNAH,126/01/04,2,404,PN,209,0.517327


In [8]:
# Merge with ethnic composition per stream; use an outer join + indicator so mismatches
# would be visible rather than silently dropped.
merged = results_long.merge(
    stream_ethnic,
    on=['dun', 'dm_code', 'saluran'],
    how='outer',
    indicator=True,
)

print(merged['_merge'].value_counts())
unmatched = merged[merged['_merge'] != 'both']
assert len(unmatched) == 0, f"{len(unmatched)} rows failed to match between results and ethnic composition"
merged = merged.drop(columns='_merge')

print(merged.shape)
merged.head(10)

_merge
both          3227
left_only        0
right_only       0
Name: count, dtype: int64
(3227, 16)


,dun,dm_code,saluran,valid_votes,party,votes,vote_share,chinese,indian,malay,other,n_registered,pct_malay,pct_chinese,pct_indian,pct_other
0,N01 CHENNAH,126/01/01,1,325,PN,165,0.507692,1,0,447,2,450,0.993333,0.002222,0.000000,0.004444
1,N01 CHENNAH,126/01/01,1,325,PH,160,0.492308,1,0,447,2,450,0.993333,0.002222,0.000000,0.004444
2,N01 CHENNAH,126/01/01,2,347,PN,229,0.659942,0,0,535,4,539,0.992579,0.000000,0.000000,0.007421
3,N01 CHENNAH,126/01/01,2,347,PH,118,0.340058,0,0,535,4,539,0.992579,0.000000,0.000000,0.007421
4,N01 CHENNAH,126/01/02,1,265,PN,73,0.275472,248,17,149,36,450,0.331111,0.551111,0.037778,0.080000
5,N01 CHENNAH,126/01/02,1,265,PH,192,0.724528,248,17,149,36,450,0.331111,0.551111,0.037778,0.080000
6,N01 CHENNAH,126/01/02,2,215,PN,76,0.353488,55,8,124,156,343,0.361516,0.160350,0.023324,0.454810
7,N01 CHENNAH,126/01/02,2,215,PH,139,0.646512,55,8,124,156,343,0.361516,0.160350,0.023324,0.454810
8,N01 CHENNAH,126/01/03,1,282,PN,98,0.347518,123,22,302,3,450,0.671111,0.273333,0.048889,0.006667
9,N01 CHENNAH,126/01/03,1,282,PH,184,0.652482,123,22,302,3,450,0.671111,0.273333,0.048889,0.006667


## A5: Regression engine — estimate Malay/Chinese support per seat per party

Per the paper's Appendix 3: for each `(dun, party)`, run two bivariate OLS regressions
across that seat's streams — `vote_share ~ pct_malay` and `vote_share ~ pct_chinese` —
and evaluate each at x=1 (a hypothetical 100%-of-that-group stream) to get the
estimated support level. Chinese is capped at 99%, everything else at 100%, and a
group is only reported if it's ≥20% of the seat's registered voters.

In [9]:
# Seat-level ethnic composition (weighted by each stream's registered voters), used
# for the paper's "only report a group if it's >=20% of registered voters" rule.
# Computed from ALL streams in the dun (independent of party/contest).
def weighted_pct(g, col):
    return (g[col] * g['n_registered']).sum() / g['n_registered'].sum()

seat_ethnic = stream_ethnic.groupby('dun').apply(
    lambda g: pd.Series({
        'malay': weighted_pct(g, 'pct_malay'),
        'chinese': weighted_pct(g, 'pct_chinese'),
        'indian': weighted_pct(g, 'pct_indian'),
        'other': weighted_pct(g, 'pct_other'),
        'n_registered': g['n_registered'].sum(),
    }),
    include_groups=False,
).reset_index()

print(seat_ethnic.shape)  # expect (36, 6)
seat_ethnic.head()

(36, 6)


,dun,malay,chinese,indian,other,n_registered
0,N01 CHENNAH,0.461915,0.443664,0.020523,0.073898,14520.0
1,N02 PERTANG,0.654610,0.199705,0.066905,0.078780,12884.0
2,N03 SUNGAI LUI,0.789754,0.061114,0.084152,0.064980,18621.0
3,N04 KLAWANG,0.650572,0.285229,0.038164,0.026035,12944.0
4,N05 SERTING,0.775773,0.126238,0.087016,0.010973,29983.0


In [10]:
import numpy as np

def estimate_support(group, ethnic_pct_col):
    """Bivariate OLS: vote_share ~ pct_<ethnic>, evaluated at x=1 (100%)."""
    if len(group) < 2 or group[ethnic_pct_col].nunique() < 2:
        return np.nan
    slope, intercept = np.polyfit(group[ethnic_pct_col], group['vote_share'], 1)
    return intercept + slope  # predicted vote_share at pct = 1.0


def cap_support(value, ethnic_group):
    if pd.isna(value):
        return value
    cap = 0.99 if ethnic_group == 'chinese' else 1.0
    return min(max(value, 0.0), cap)


def run_regression_engine(merged_df, seat_ethnic_df, threshold=0.20):
    """For every (dun, party), estimate Malay and Chinese support via A5's bivariate
    OLS method, applying the paper's cap and >=20%-of-registered-voters conventions."""
    records = []
    for (dun, party), group in merged_df.groupby(['dun', 'party']):
        seat_row = seat_ethnic_df.loc[seat_ethnic_df['dun'] == dun].iloc[0]
        row = {'dun': dun, 'party': party, 'n_streams': len(group)}
        for ethnic_group, pct_col in [('malay', 'pct_malay'), ('chinese', 'pct_chinese')]:
            estimate = cap_support(estimate_support(group, pct_col), ethnic_group)
            if seat_row[ethnic_group] < threshold:
                estimate = np.nan
            row[f'{ethnic_group}_support'] = estimate
        records.append(row)
    return pd.DataFrame(records)


support_table = run_regression_engine(merged, seat_ethnic)
print(support_table.shape)
support_table.head(20)

(83, 5)


,dun,party,n_streams,malay_support,chinese_support
0,N01 CHENNAH,PH,28,0.297840,0.926431
1,N01 CHENNAH,PN,28,0.702160,0.073569
2,N02 PERTANG,BN,26,0.552833,NaN
3,N02 PERTANG,PN,26,0.447167,NaN
4,N03 SUNGAI LUI,BN,41,0.455297,NaN
5,N03 SUNGAI LUI,PN,41,0.544703,NaN
6,N04 KLAWANG,IND,27,0.061701,0.002201
7,N04 KLAWANG,PH,27,0.331755,0.970247
8,N04 KLAWANG,PN,27,0.606544,0.027552
9,N05 SERTING,BN,39,0.386563,NaN


## A6: Validate against the paper's published DAP numbers

Filter `support_table` to `party == 'PH'` and the 11 seats in `dap_seats.txt`, then
compare against the paper's Table 6 "Voting PH — PRN 2023" Malay/Chinese columns
(hardcoded below, transcribed from the PDF).

In [11]:
import re

dap_seats = pd.read_csv('dap_seats.txt', sep='\t', header=None,
                         names=['parlimen', 'dun', 'candidate', 'party'])
for col in dap_seats.columns:
    dap_seats[col] = dap_seats[col].str.strip()

# Normalize dun the same way script.py normalized the rolls: "N.01 Chennah" -> "N01 CHENNAH"
dap_seats['dun'] = dap_seats['dun'].apply(
    lambda v: re.sub(r'^([A-Za-z]+)\.(\d+)', r'\1\2', v).upper()
)

print(dap_seats.shape)  # expect (11, 4)
assert dap_seats['dun'].isin(seat_ethnic['dun']).all(), "some dap_seats DUNs didn't normalize to a known seat"
dap_seats[['dun', 'candidate']]

(11, 4)


,dun,candidate
0,N01 CHENNAH,LOKE SIEW FOOK
1,N08 BAHAU,TEO KOK SEONG
2,N10 NILAI,ARUL KUMAR A/L JAMBUNATHAN
3,N11 LOBAK,CHEW SEH YONG
4,N12 TEMIANG,NG CHIN TSAI
5,N21 BUKIT KEPAYANG,NICOLE TAN LEE KOON
6,N22 RAHANG,SIAU MEOW KONG
7,N23 MAMBAU,YAP YEW WENG
8,N24 SEREMBAN JAYA,GUNASEKAREN A/L PALASAMY
9,N30 LUKUT,CHOO KEN HWA


In [12]:
# Paper's Table 6, "Voting PH - PRN 2023" columns, NS rows only (transcribed from the PDF)
paper_table6_ns = pd.DataFrame([
    ('N01 CHENNAH',           31.3, 92.6),
    ('N08 BAHAU',             36.3, 99.0),
    ('N10 NILAI',             33.2, 99.0),
    ('N11 LOBAK',             np.nan, 98.9),
    ('N12 TEMIANG',           21.5, 98.3),
    ('N21 BUKIT KEPAYANG',    36.9, 99.0),
    ('N22 RAHANG',            34.6, 99.0),
    ('N23 MAMBAU',            np.nan, 99.0),
    ('N24 SEREMBAN JAYA',     28.4, 99.0),
    ('N30 LUKUT',             25.1, 99.0),
    ('N36 REPAH',             29.8, 99.0),
], columns=['dun', 'paper_malay', 'paper_chinese'])
paper_table6_ns['paper_malay'] /= 100
paper_table6_ns['paper_chinese'] /= 100

ours = support_table[(support_table['party'] == 'PH') & (support_table['dun'].isin(dap_seats['dun']))]

comparison = paper_table6_ns.merge(ours, on='dun', how='left')
comparison['malay_diff_pp'] = (comparison['malay_support'] - comparison['paper_malay']) * 100
comparison['chinese_diff_pp'] = (comparison['chinese_support'] - comparison['paper_chinese']) * 100

assert len(comparison) == 11, f"expected 11 DAP seats, got {len(comparison)}"
comparison[['dun', 'paper_malay', 'malay_support', 'malay_diff_pp',
            'paper_chinese', 'chinese_support', 'chinese_diff_pp']]

,dun,paper_malay,malay_support,malay_diff_pp,paper_chinese,chinese_support,chinese_diff_pp
0,N01 CHENNAH,0.313,0.297840,-1.516015,0.926,0.926431,0.043051
1,N08 BAHAU,0.363,0.358219,-0.478123,0.990,0.990000,0.000000
2,N10 NILAI,0.332,0.270553,-6.144666,0.990,0.990000,0.000000
3,N11 LOBAK,NaN,NaN,NaN,0.989,0.989229,0.022928
4,N12 TEMIANG,0.215,0.175083,-3.991679,0.983,0.982628,-0.037156
5,N21 BUKIT KEPAYANG,0.369,0.374109,0.510914,0.990,0.990000,0.000000
6,N22 RAHANG,0.346,0.326921,-1.907933,0.990,0.990000,0.000000
7,N23 MAMBAU,NaN,NaN,NaN,0.990,0.990000,0.000000
8,N24 SEREMBAN JAYA,0.284,0.186831,-9.716865,0.990,0.990000,0.000000
9,N30 LUKUT,0.251,0.234139,-1.686084,0.990,0.990000,0.000000
